# Deliverable D2: Data Cleaning & Processing

This notebook demonstrates our data cleaning rules: parsing dates, sorting records, forward-filling weekend and holiday gaps in Net Asset Value series, and standardizing transaction properties.

In [1]:
import os
import pandas as pd
import numpy as np

raw_dir = os.path.join("..", "data", "raw")
processed_dir = os.path.join("..", "data", "processed")

### 1. Cleaning nav_history - Implementing Forward-Fill for weekends/holidays

In [2]:
df_nav = pd.read_csv(os.path.join(raw_dir, "02_nav_history.csv"))
print("Original rows count:", len(df_nav))

df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'])
df_nav = df_nav[df_nav['nav'] > 0]

cleaned_groups = []
for amfi_code, group in df_nav.groupby('amfi_code'):
    group = group.sort_values('date')
    min_date = group['date'].min()
    max_date = group['date'].max()
    
    full_range = pd.date_range(start=min_date, end=max_date, freq='D')
    group = group.set_index('date').reindex(full_range)
    group['amfi_code'] = amfi_code
    group['nav'] = group['nav'].ffill()
    group = group.reset_index().rename(columns={'index': 'date'})
    cleaned_groups.append(group)
    
df_nav_cleaned = pd.concat(cleaned_groups, ignore_index=True)
print("Cleaned & daily forward-filled rows count:", len(df_nav_cleaned))
df_nav_cleaned.head(10)

Original rows count: 46000
Cleaned & daily forward-filled rows count: 64320


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639
5,2022-01-08,100016,515.1639
6,2022-01-09,100016,515.1639
7,2022-01-10,100016,510.7136
8,2022-01-11,100016,513.5542
9,2022-01-12,100016,512.3195


### 2. Standardizing transaction type values

In [3]:
df_tx = pd.read_csv(os.path.join(raw_dir, "08_investor_transactions.csv"))
print("Unique transaction_types before:", df_tx['transaction_type'].unique())

df_tx['transaction_type'] = df_tx['transaction_type'].astype(str).str.strip()
mapping = {
    'sip': 'SIP', 'Sip': 'SIP', 'SIP': 'SIP',
    'lumpsum': 'Lumpsum', 'Lumpsum': 'Lumpsum',
    'redemption': 'Redemption', 'Redemption': 'Redemption'
}
df_tx['transaction_type'] = df_tx['transaction_type'].map(lambda x: mapping.get(x, x))
print("Unique transaction_types after:", df_tx['transaction_type'].unique())

Unique transaction_types before: <StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
Unique transaction_types after: <StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
